In [4]:
##################################################################
# # ! Two methods to approximate a sphere with a polyhedron:
# (1) Recursive subdivision of an icosahedron
# (2) Triangulated regular geographic grid
##################################################################
# %%
# # ! Setup
import pyvista as pv
import numpy as np
from gravity_forward_numpy import spherical_edge_length_range

In [5]:
# # ! Recursive subdivision of an icosahedron

# True volume and surface area of unit sphere
VOL_TRUE = 4 * np.pi / 3
AREA_TRUE = 4 * np.pi

nsub_max = 10
NSUBs = np.arange(nsub_max)

# ------------------------------------------------------------------
# (1) Recursive subdivision of an icosahedron
# ------------------------------------------------------------------
print("\n" + "="*68)
print("Recursive subdivision of an icosahedron")
print(f"{'N':>4} {'Nv':>8} {'Nf':>8} {'Vol_err':>10} {'Area_err':>10} {'Min_psi':>10} {'Max_psi':>10}")
print("-" * 68)

for nsub in NSUBs:
    mesh = pv.Icosphere(radius=1.0, nsub=nsub)
    verts = mesh.points
    faces = mesh.regular_faces

    nv = verts.shape[0]
    nf = faces.shape[0]
    vol_err = 1.0 - mesh.volume / VOL_TRUE
    area_err = 1.0 - mesh.area / AREA_TRUE

    min_psi, max_psi = spherical_edge_length_range(verts, faces)
    min_psi_arcmin = 60.0 * np.rad2deg(min_psi)
    max_psi_arcmin = 60.0 * np.rad2deg(max_psi)

    print(f"{nsub:>4} {nv:>8} {nf:>8} "
          f"{vol_err:>10.2e} {area_err:>10.2e} "
          f"{min_psi_arcmin:>10.3f} {max_psi_arcmin:>10.3f}")


Recursive subdivision of an icosahedron
   N       Nv       Nf    Vol_err   Area_err    Min_psi    Max_psi
--------------------------------------------------------------------
   0       12       20   3.95e-01   2.38e-01   3806.097   3806.097
   1       42       80   1.27e-01   7.17e-02   1903.048   2160.000
   2      162      320   3.38e-02   1.89e-02    872.726   1121.964
   3      642     1280   8.62e-03   4.79e-03    410.913    566.657
   4     2562     5120   2.17e-03   1.20e-03    198.831    284.052
   5    10242    20480   5.42e-04   3.01e-04     97.751    142.117
   6    40962    81920   1.36e-04   7.52e-05     48.459     71.070
   7   163842   327680   3.39e-05   1.88e-05     24.126     35.536
   8   655362  1310720   8.47e-06   4.70e-06     12.037     17.768
   9  2621442  5242880   2.12e-06   1.18e-06      6.012      8.884


In [6]:
# # ! Triangulated regular geographic grid

# ------------------------------------------------------------------
# (2) Triangulated regular geographic grid
# ------------------------------------------------------------------
print("\n" + "="*68)
print("Triangulated Regular Geographic Grid")
print(f"{'N':>4} {'Nv':>8} {'Nf':>8} {'Vol_err':>10} {'Area_err':>10} {'Min_psi':>10} {'Max_psi':>10}")
print("-" * 68)

Lon_Res = 5 * (2 ** NSUBs)
Lat_Res = (2 ** (NSUBs + 1)) + 2

for i, nsub in enumerate(NSUBs):
    theta_res = Lon_Res[i]
    phi_res   = Lat_Res[i]
    mesh = pv.Sphere(radius=1.0, theta_resolution=theta_res, phi_resolution=phi_res)
    verts = mesh.points
    faces = mesh.regular_faces

    nv = verts.shape[0]
    nf = faces.shape[0]
    vol_err = 1.0 - mesh.volume / VOL_TRUE
    area_err = 1.0 - mesh.area / AREA_TRUE

    min_psi, max_psi = spherical_edge_length_range(verts, faces)
    min_psi_arcmin = 60.0 * np.rad2deg(min_psi)
    max_psi_arcmin = 60.0 * np.rad2deg(max_psi)

    print(f"{nsub:>4} {nv:>8} {nf:>8} "
          f"{vol_err:>10.2e} {area_err:>10.2e} "
          f"{min_psi_arcmin:>10.3f} {max_psi_arcmin:>10.3f}")


Triangulated Regular Geographic Grid
   N       Nv       Nf    Vol_err   Area_err    Min_psi    Max_psi
--------------------------------------------------------------------
   0       12       20   4.32e-01   2.46e-01   3600.000   5462.699
   1       42       80   1.54e-01   8.02e-02   1255.805   3029.140
   2      162      320   4.60e-02   2.33e-02    368.040   1610.749
   3      642     1280   1.26e-02   6.31e-03     99.126    833.287
   4     2562     5120   3.29e-03   1.65e-03     25.659    424.208
   5    10242    20480   8.41e-04   4.20e-04      6.522    214.076
   6    40962    81920   2.13e-04   1.06e-04      1.644    107.542
   7   163842   327680   5.34e-05   2.67e-05      0.413     53.898
   8   655362  1310720   1.34e-05   6.69e-06      0.103     26.981
   9  2621442  5242880   3.35e-06   1.68e-06      0.026     13.499


In [7]:
# # ! Plot

pl = pv.Plotter(shape=(2, 3), image_scale=3)
pickNs = [2, 3, 4]

# Top row: Icosphere
for col, nsub in enumerate(pickNs):
    mesh = pv.Icosphere(radius=1.0, nsub=nsub)
    mesh = mesh.compute_cell_sizes()
    areas = mesh['Area']
    mesh[f'Area (%) nsub={nsub}'] = 100 * areas / areas.sum()
    pl.subplot(0, col)
    pl.add_mesh(mesh, scalars=f'Area (%) nsub={nsub}', cmap='viridis',
                scalar_bar_args=dict(title_font_size=14, label_font_size=12,
                                     n_labels=3, position_y=0.05, fmt='%.3f'))
    pl.camera.zoom(1.25)

# Bottom row: Geographic grid
for col, nsub in enumerate(pickNs):
    mesh = pv.Sphere(radius=1.0,
                     theta_resolution=Lon_Res[nsub],
                     phi_resolution=Lat_Res[nsub])
    mesh = mesh.compute_cell_sizes()
    areas = mesh['Area']
    mesh[f'Area (%) level={nsub}'] = 100 * areas / areas.sum()
    pl.subplot(1, col)
    pl.add_mesh(mesh, scalars=f'Area (%) level={nsub}', cmap='viridis',
                scalar_bar_args=dict(title_font_size=14, label_font_size=12,
                                     n_labels=3, position_y=0.05, fmt='%.3f'))
    pl.camera.zoom(1.25)

pl.show()
pl.screenshot("icosphere_geographic.png");

Widget(value='<iframe src="http://localhost:44569/index.html?ui=P_0x787d16305610_1&reconnect=auto" class="pyvi…